In [ ]:
import tensorflow as tf
import numpy as np
from PIL import Image
import cv2
import os

class ESPCNInference:
    def __init__(self, model_path=None, scale_size=2, patch_size=10):
        self.scale_size = scale_size  
        self.patch_size = patch_size
        self.model = None
        if model_path:
            self.load_model(model_path)
        else:

            input_shape = (patch_size, patch_size, 3)
            self.model = build_espcn_model_with_skip_connections(input_shape, scale_size)
    
    def load_model(self, model_path):

        try:
            self.model = tf.keras.models.load_model(
                model_path,
                custom_objects={
                    'DepthToSpace': DepthToSpace,
                    'SinglePixelAttention': SinglePixelAttention,
                    'psnr': psnr
                }
            )
            print("Model loaded successfully")
        except Exception as e:
            print(f"Error loading model: {str(e)}")
            raise
    
    def save_model(self, model_path):

        self.model.save(model_path)
        print(f"Model saved to {model_path}")
    
    def preprocess_image(self, image_path):
        try:
            
            img = Image.open(image_path).convert('RGB')
            img_array = np.array(img, dtype=np.uint8)
            img_yuv = cv2.cvtColor(img_array, cv2.COLOR_RGB2YUV)
            
            #
            img_yuv = img_yuv.astype(np.float32) / 255.0
            
            
            img_yuv = np.expand_dims(img_yuv, axis=0)
            
            return img_yuv, img_array.shape[:2]
            
        except Exception as e:
            print(f"Error preprocessing image: {str(e)}")
            raise
    
    def create_patches(self, image, include_padding=True):

        h, w = image.shape[1:3]
        
        if include_padding:
            
            pad_h = (self.patch_size - h % self.patch_size) % self.patch_size
            pad_w = (self.patch_size - w % self.patch_size) % self.patch_size
            
            
            if pad_h > 0 or pad_w > 0:
                image = tf.pad(image, [[0, 0], [0, pad_h], [0, pad_w], [0, 0]], mode='REFLECT')
        
        
        patches = tf.image.extract_patches(
            images=image,
            sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            rates=[1, 1, 1, 1],
            padding='VALID'
        )
        
        
        patches = tf.reshape(patches, [-1, self.patch_size, self.patch_size, 3])
        return patches, image.shape[1:3]
    
    def reconstruct_image(self, patches, original_size, padded_size):

        
        h, w = padded_size
        patch_size_hr = self.patch_size * self.scale_size  
        patches_per_row = w // self.patch_size
        patches_per_col = h // self.patch_size
        
        
        patches = tf.reshape(patches, [patches_per_col, patches_per_row, 
                                     patch_size_hr, patch_size_hr, 3])
        
        
        image = tf.transpose(patches, [0, 2, 1, 3, 4])
        image = tf.reshape(image, [1, h * self.scale_size, w * self.scale_size, 3])
        
        
        orig_h, orig_w = original_size
        image = image[:, :orig_h * self.scale_size, :orig_w * self.scale_size, :]
        
        return image
    
    def postprocess_image(self, image):

        image = tf.clip_by_value(image, 0, 1)
        image = tf.round(image * 255.0)
        image = tf.cast(image, tf.uint8)
        image = cv2.cvtColor(image[0].numpy(), cv2.COLOR_YUV2RGB)
        return image
    
    def upscale_image(self, image_path, output_path=None):
      
        try:

            img_yuv, original_size = self.preprocess_image(image_path)
            
            patches, padded_size = self.create_patches(img_yuv)
            
           
            sr_patches = self.model.predict(patches, verbose=0)
            
           
            sr_image = self.reconstruct_image(sr_patches, original_size, padded_size)
            
           
            final_image = self.postprocess_image(sr_image)
            
            if output_path:
                Image.fromarray(final_image).save(output_path)
                print(f"Saved super-resolved image to {output_path}")
            
            return final_image
            
        except Exception as e:
            print(f"Error during upscaling: {str(e)}")
            raise
    
    def upscale_directory(self, input_dir, output_dir):

        os.makedirs(output_dir, exist_ok=True)
        
        for filename in os.listdir(input_dir):
            if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
                input_path = os.path.join(input_dir, filename)
                output_path = os.path.join(output_dir, f"sr_{filename}")
                
                try:
                    self.upscale_image(input_path, output_path)
                    print(f"Processed {filename}")
                except Exception as e:
                    print(f"Error processing {filename}: {str(e)}")
                    continue

# Example usage:
"""
# After training your model:
inference_model = ESPCNInference(scale_size=2, patch_size=10)  # Changed scale_size to 2

# Transfer weights from trained model
inference_model.model.set_weights(espcn_model_with_skip_and_attention.get_weights())

# Save the model if needed
inference_model.save_model('espcn_model.h5')

# Upscale a single image
upscaled_image = inference_model.upscale_image('input.jpg', 'output.jpg')

# Or upscale all images in a directory
inference_model.upscale_directory('input_dir', 'output_dir')
"""